# PyMeshIt Headless Batch Workflow

This notebook shows how to run the new PyMeshIt headless workflow without opening the GUI.

Use it for simple batch cases where the GUI workflow would normally be:

1. Load surface and optional well files.
2. Compute hulls, segments, initial triangulations, and intersections.
3. In the Refine & Mesh tab, choose **select intersections**.
4. Generate conforming surface meshes.
5. Run TetGen and export the final mesh.

The equivalent API option is `constraint_mode="intersections"`. With `preserve_boundary_hulls=True`, PyMeshIt keeps each surface hull as the PLC boundary and adds selected intersection lines inside it, matching the GUI benchmark workflow.

## 1. Install or Import PyMeshIt

If you installed PyMeshIt from PyPI after the headless API release, the normal import should work. If you are running this notebook from the GitHub repository, use the local repository copy or install the repository in editable mode first.

Important: inside Jupyter, use `%pip install ...`. Do not type `pip3 install ...` as plain Python; that command only works in a terminal. After installing into the notebook kernel, restart the kernel and run the notebook again.

If you edit `Pymeshit/headless.py` while this notebook is already open, rerun the import cell below or restart the kernel. Jupyter otherwise keeps the old module in memory.

In [ ]:
# Run ONE of these only if the imports below fail in a fresh environment.
# Keep the leading % when running inside Jupyter.

# If this notebook is opened from the cloned GitHub repository:
# %pip install -e ..

# If you want the PyPI release instead:
# %pip install pymeshit

# Terminal-only equivalent, not valid as plain Python in a notebook cell:
# pip3 install pymeshit

In [ ]:
from pathlib import Path
import sys
import numpy as np


def find_repo_root(start: Path):
    """Find a local MeshIt checkout that contains Pymeshit/headless.py."""
    for candidate in [start, *start.parents]:
        if (candidate / "Pymeshit" / "headless.py").exists():
            return candidate
    return None


def path_is_under(path: Path, root: Path) -> bool:
    try:
        path.resolve().relative_to(root.resolve())
        return True
    except ValueError:
        return False


# Prefer this source checkout over an older installed Pymeshit package.
# Also force-reload Pymeshit.headless from disk, because Jupyter caches modules.
REPO_ROOT = find_repo_root(Path.cwd())
if REPO_ROOT is not None:
    sys.path.insert(0, str(REPO_ROOT))
    existing = sys.modules.get("Pymeshit")
    if existing is not None and getattr(existing, "__file__", None):
        if not path_is_under(Path(existing.__file__), REPO_ROOT):
            for module_name in list(sys.modules):
                if module_name == "Pymeshit" or module_name.startswith("Pymeshit."):
                    del sys.modules[module_name]
        else:
            sys.modules.pop("Pymeshit.headless", None)
    else:
        sys.modules.pop("Pymeshit.headless", None)

try:
    import Pymeshit.headless as headless_module
    from Pymeshit.headless import (
        MeshCase,
        MeshOptions,
        SurfaceSpec,
        WellSpec,
        MaterialSpec,
        read_points,
        run_mesh_case,
    )
except ImportError as exc:
    import Pymeshit
    raise ImportError(
        "This Python kernel is importing an older PyMeshIt package from:\n"
        f"  {getattr(Pymeshit, '__file__', 'unknown')}\n\n"
        "Install the current repository into this notebook kernel with:\n"
        "  %pip install -e ..\n\n"
        "Then restart the kernel and run the notebook again."
    ) from exc

print("PyMeshIt headless API imported successfully")
print("Using headless module:", Path(headless_module.__file__).resolve())
if REPO_ROOT is not None:
    print("Using source checkout:", REPO_ROOT)


## 2. Put Input Files in One Folder

Accepted point inputs are text/CSV-style files with columns `x y z`, or `.vtu/.vtk/.vtp` files readable by PyVista.

Recommended naming convention:

- outer/boundary surfaces: `bottom.txt`, `top.txt`, `xmin.txt`, etc.
- geological layer/internal surfaces: `layer_1.txt`, `layer_2.txt`, etc.
- faults: `fault_1.txt`, `fault_2.txt`, etc.
- wells: `well_1.txt`, `well_2.txt`, etc.

For a volume mesh, the `border` surfaces must define a closed outer domain. Internal layers should usually be `unit`; fault surfaces should be `fault`.

In [ ]:
# Option A: local Jupyter / VS Code / JupyterLab
# This repo includes a small example dataset in examples/data.
# If you run the notebook from the repository root, this path works directly.
DEFAULT_DATA_DIR = Path("examples/data")
DATA_DIR = DEFAULT_DATA_DIR if DEFAULT_DATA_DIR.exists() else Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Option B: Google Colab upload
# Uncomment this block in Colab, then upload your files.
# from google.colab import files
# uploaded = files.upload()
# for filename, content in uploaded.items():
#     (DATA_DIR / filename).write_bytes(content)

print("Data folder:", DATA_DIR.resolve())
print("Files:")
for path in sorted(DATA_DIR.glob("*")):
    print(" -", path.name)


## 3. Define Surfaces and Roles

The example below uses the files in `examples/data`. You can add any number of files to each group.

Roles:

- `border`: outer boundary of the 3D domain. These are kept even in intersection-only mode.
- `unit`: internal stratigraphic/layer surfaces. You can have multiple unit surfaces.
- `fault`: internal 2D fault/fracture surfaces. You can have multiple fault surfaces. These become surface constraints, not volumetric material regions.

Each group has one `target_size`. Smaller values make denser meshes. You can also split files into separate groups if different surfaces need different sizes.


In [ ]:
# Example input using examples/data.
# Add/remove filenames in each group for your own model.
surface_groups = [
    {
        "role": "border",
        "target_size": 15.0,
        "files": [
            "bottom.dat",
            "top.dat",
            "border1.txt",
            "border2.txt",
            "border3.txt",
            "border4.txt",
        ],
    },
    {
        "role": "unit",
        "target_size": 15.0,
        "files": [
            "middle.dat",
            # Add more unit/layer files here, for example: "layer_2.dat"
        ],
    },
    {
        "role": "fault",
        "target_size": 15.0,
        "files": [
            "fault1.txt",
            "fault2.txt",
        ],
    },
]


def build_surfaces_from_groups(surface_groups, data_dir: Path):
    surfaces = []
    for group in surface_groups:
        role = group["role"]
        target_size = float(group.get("target_size", 15.0))
        for filename in group["files"]:
            path = data_dir / filename
            if not path.exists():
                print(f"WARNING: missing {path}. Edit surface_groups or upload the file.")
                continue
            pts = read_points(path)
            name = path.stem
            print(f"{name:20s} role={role:7s} target={target_size:6.2f} points={len(pts):5d} path={path}")
            surfaces.append(SurfaceSpec(name=name, points=pts, role=role, target_size=target_size))
    return surfaces


surfaces = build_surfaces_from_groups(surface_groups, DATA_DIR)
print("Loaded surfaces:", len(surfaces))


## 4. Optional Wells

Wells are 1D polylines. If you do not have wells, leave `well_inputs` empty.

In [ ]:
well_inputs = [
    # name, filename, target_size
    # ("well_1", "well_1.txt", 20.0),
]

wells = []
for name, filename, target_size in well_inputs:
    path = DATA_DIR / filename
    if not path.exists():
        print(f"WARNING: missing {path}. Edit well_inputs or upload the file.")
        continue
    pts = read_points(path)
    print(f"{name:20s} points={len(pts)} path={path}")
    wells.append(WellSpec(name=name, points=pts, target_size=target_size))

print("Loaded wells:", len(wells))

## 5. Define Material Seed Points

Each material seed point must be inside the volume region it represents. For a two-layer model, add one seed point in each layer.

If you define no materials, PyMeshIt uses one default region seed near the PLC center. For geological models, explicit seed points are safer.

In [ ]:
# Example material seed points for examples/data.
# Put each seed inside the volume region it should label.
# With middle.dat near z=0, these two seeds label lower and upper domains.
materials = [
    MaterialSpec("lower_unit", [0.0, 0.0, -25.0], attribute=1),
    MaterialSpec("upper_unit", [0.0, 0.0, 25.0], attribute=2),
]

for material in materials:
    print(material)


## 6. Run the Workflow

`constraint_mode="intersections"` is the notebook equivalent of the GUI Select intersections button: PyMeshIt keeps each surface hull as the PLC boundary and adds selected intersection lines inside it.

Keep `preserve_boundary_hulls=True` for volume meshes. If you only want conforming surface meshes, set `generate_volume=False`.

For the benchmark data, the fault surfaces should report `constraint_source="selected"`. `hull_fallback` is only a safety fallback for cases where selected constraints still cannot be triangulated.

In [ ]:
if not surfaces:
    raise RuntimeError("No surfaces loaded. Upload files and edit surface_groups first.")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

options = MeshOptions(
    target_size=15.0,
    min_angle=20.0,
    gradient=2.0,
    hull_method="delaunay",
    interpolation="Thin Plate Spline (TPS)",
    smoothing=0.0,
    constraint_mode="intersections",
    preserve_boundary_hulls=True,
    include_hull_fallback_faults_in_volume=False,
    tetgen_switches="pq1.414aAY",
    generate_volume=True,
)

case = MeshCase(
    surfaces=surfaces,
    wells=wells,
    materials=materials,
    options=options,
)

result = run_mesh_case(case, output_path=OUTPUT_DIR / "example_data_case.vtu")

print("Failures:", result.failures)
if result.tetra_mesh is not None:
    print("Tetra mesh points:", result.tetra_mesh.n_points)
    print("Tetra mesh cells:", result.tetra_mesh.n_cells)
    print("Saved:", (OUTPUT_DIR / "example_data_case.vtu").resolve())
else:
    print("No tetra mesh was generated. Check failures and logs above.")


## 7. Inspect or Save Intermediate Surface Meshes

The result also contains conforming surface meshes. These are useful for checking whether the intersection-only constraint selection worked before TetGen.

In [ ]:
for surface_idx, data in result.conforming_surface_data.items():
    print(
        surface_idx,
        data["name"],
        "source=", data.get("constraint_source", "selected"),
        "vertices=", len(data["vertices"]),
        "triangles=", len(data["triangles"]),
    )

Optional PyVista visualization. This works best in local Jupyter. In some remote notebooks you may need off-screen rendering or skip plotting.

In [ ]:
# Optional visualization
# import pyvista as pv
# if result.tetra_mesh is not None:
#     scalars = "MaterialID" if "MaterialID" in result.tetra_mesh.cell_data else None
#     result.tetra_mesh.plot(scalars=scalars, show_edges=True)

## 8. Batch Run Multiple Cases

For parameter studies, create one folder or file set per geometry case, then run the same function in a loop. The example below assumes each case has the same filenames in different folders.

In [ ]:
def build_case_from_folder(case_dir: Path, case_name: str) -> MeshCase:
    local_surfaces = build_surfaces_from_groups(surface_groups, case_dir)
    if not local_surfaces:
        raise RuntimeError(f"No surfaces loaded for {case_name} from {case_dir}")

    return MeshCase(
        surfaces=local_surfaces,
        wells=[],
        materials=materials,
        options=options,
    )


# Example batch structure: each folder contains the same filenames listed in surface_groups.
# data/case_dip_30/bottom.dat, top.dat, border1.txt, ..., fault1.txt
# data/case_dip_45/bottom.dat, top.dat, border1.txt, ..., fault1.txt
batch_cases = [
    # ("dip_30", DATA_DIR / "case_dip_30"),
    # ("dip_45", DATA_DIR / "case_dip_45"),
    # ("dip_60", DATA_DIR / "case_dip_60"),
]

for case_name, case_dir in batch_cases:
    print("Running", case_name)
    batch_case = build_case_from_folder(case_dir, case_name)
    batch_result = run_mesh_case(batch_case, output_path=OUTPUT_DIR / f"{case_name}.vtu")
    print("  failures:", batch_result.failures)
    if batch_result.tetra_mesh is not None:
        print("  cells:", batch_result.tetra_mesh.n_cells)


## Troubleshooting

- `No surfaces loaded`: upload files or fix `surface_inputs` filenames.
- `No conforming surface meshes`: the selected constraints do not form enough PLC geometry. Try `constraint_mode="all"` first, or keep `preserve_boundary_hulls=True`.
- TetGen fails: check that `border` surfaces form a closed domain and material seed points are inside valid regions.
- Wrong material IDs: move material seed points away from faults, holes, or boundaries.
- Too coarse/fine mesh: change `target_size` globally or per surface.
- Only need surface meshes: set `generate_volume=False` in `MeshOptions`.